# Profils des enquêtés

## Les pratiques de travail

In [2]:
import pandas as pd
import re
import os #pour naviguer dans les dossiers
from io import StringIO
import s3fs #pour connecter au bucket
import scipy.stats

import seaborn as sns
import matplotlib.pyplot as plt
import numpy as np

In [3]:


# Create filesystem object
S3_ENDPOINT_URL = "https://" + os.environ["AWS_S3_ENDPOINT"]
fs = s3fs.S3FileSystem(client_kwargs={'endpoint_url': S3_ENDPOINT_URL})

BUCKET_OUT = "aluneau"

last_file="anonymous_answer_2026-07-24.csv"


In [4]:
with fs.open(f"{BUCKET_OUT}/data_eP8/raw/{last_file}", "r") as file_in:
    df0 = pd.read_csv(file_in, sep =",")

list_affil = pd.read_csv("../list_affiliation.csv", sep =",")
df0 = df0.loc[~df0.q45_clé.isin(["9EAP-NB4B","QNYZ-3MH2"])].merge(list_affil, on = ["q45_clé", "q44_ufr_labo"], how = "left")

In [5]:

df_col = pd.read_csv("../../le_questionnaire/dico_variable.csv", sep = ",")


In [6]:
list_nominal_simple = [x for x in df_col.label.loc[(df_col.type.isin(["simple_nominal", "booléen","ordinal"]))]]


In [7]:
def split_multiple_choices(data, column, index, sep = '|'):
    """
    split and explode column with multiple value

    data = a panda dataframe
    column = name of column which we want to split.
    index = column corresponding to id of rows
    sep = by default '|'. Character use to separate values.
    """
    df_split = data.copy()
    df_split[column]= df_split.apply(lambda row: row[column].replace(";",sep) ,1 )
    df_split[column] = df_split[column].str.split(sep)
    df_explode = df_split.explode(column)
    gb_data = df_explode.groupby([column]).agg(nb = (index, "size")).sort_values("nb", ascending=False).reset_index()
    gb_data["total"] = data[index].nunique()
    gb_data["freq"] = gb_data.nb/gb_data.total*100
    gb_data["total_freq"] = gb_data.total/gb_data.total*100
    
    return df_explode, gb_data
    

In [8]:
def grouped_question(data, column, index = "q45_clé"):
    """


    data = a panda dataframe
    column = name of column which we want to split.
    index = column corresponding to id of rows
    sep = by default '|'. Character use to separate values.
    """
    df_tmp = data.copy()
    gb_data = df_tmp.groupby([column]).agg(nb = (index, "size")).reset_index()
    gb_data["total"] = data[index].nunique()
    gb_data["freq"] = gb_data.nb/gb_data.total*100
    gb_data["total_freq"] = gb_data.total/gb_data.total*100

    return gb_data

In [9]:
def tab_croise(data, x, y, regroup_y = True, khideux = False) :
    """
    x = variable en ligne
    y = variable en colonne
    """
    if regroup_y == True:

        data.loc[data[y].str.lower().str.contains("oui"), f"{y}_rec"] = "Oui"
        data.loc[data[y].str.lower().str.contains("non"), f"{y}_rec"] = "Non"

        cross_tab = pd.crosstab(data[x], data[f"{y}_rec"], margins = True, margins_name="Total", normalize=False)
        cross_tab0 = pd.crosstab(data[x], data[f"{y}_rec"], margins = False)
    else:
        cross_tab = pd.crosstab(data[x], data[y], margins = True, margins_name="Total", normalize=False)
        cross_tab0 = pd.crosstab(data[x], data[y], margins = False, normalize=False)

    if khideux == True:
        print(scipy.stats.chi2_contingency(cross_tab0))
        st_chi2, st_p, st_dof, st_exp = scipy.stats.chi2_contingency(cross_tab0)
        chi2 = pd.DataFrame(data={"stats":["chi2","df","p-value"], "values":[st_chi2, st_dof, st_p]})
        cross_tab = pd.concat([cross_tab, chi2])
    else:
        pass

    return cross_tab.fillna("").reset_index(), cross_tab0

In [10]:
df_exp, gb_data = split_multiple_choices(df0[["q45_clé","q26_env_travail"]], "q26_env_travail", "q45_clé", sep = '|')
print(gb_data[["q26_env_travail","nb","freq","total"]].to_markdown(index=False))

| q26_env_travail          |   nb |    freq |   total |
|:-------------------------|-----:|--------:|--------:|
| A mon domicile           |  108 | 90.7563 |     119 |
| Dans mon bureau sur site |   70 | 58.8235 |     119 |
| Autre, précisez          |   31 | 26.0504 |     119 |


La première question interrogeant les pratiques de travail des enquêtés porte sur le lieu habituel de travail en classant par ordre de priorité les options suivantes :
- "À mon domicile"
- "Dans mon bureau sur site"
- "Autre"

90% des personnes interrogés disent travailler le plus souvent à leur domicile ({numref}`env_travail`). C'est aussi le lieu le plus souvent cité en premier. 71% des répondants travaillent ainsi prioritairement à leur domicile. Tandis que le bureau est en moyenne classé deuxième dans l'ordre des priorités.  

Parmis les autres lieux cités, on trouve tout d'abord les bibliothèques et les autres sites (CNRS, Campus Condorcet, partenaires de recherche). On note également que plusieurs répondants ont indiqué qu'ils travaillaient un peu partout, là où ils pouvaient faute de bureaux disponibles sur le campus. 

> "Où je peux, puisque nous n'avon pas de bureaux"

> "bureau de collègues dans d'autres universités"

> "Bureau MSH OU BIBLIOTHÈQUE MAIS JE MANQUE CRUELLEMENT D'UN ESPACE DE TRAVAIL"

> "où je peux, je n'ai pas de bureau à la fac (15 m2 pour 6 titulaires)"


Ces réactions recoupent les résultats observés concernant les services à développer où la construction d'espace de travail avaient été choisi par plus de 80% des répondants ({numref}`11_service_to_develop`).


```{table} Où travaillez-vous le plus souvent ?
:name: env_travail


|   rang |   A mon domicile         |   Dans mon bureau sur site         |   Autre, précisez         |
|-------:|-------------------------:|-----------------------------------:|--------------------------:|
|      1 |               85 (71,4%) |                         29 (24,4%) |                 5  (4,2%) |
|      2 |               20 (16,8%) |                         36 (20,3%) |                12 (10,1%) |
|      3 |                3  (2,5%) |                          5  (4,2%) |                14 (11,8%) |
|Total   |              108 (90,8%) |                         70 (58,8%) |                31 (26,1%) |

```

Les systèmes d'exploitation les plus répandus sont Apple et Windows. Seules 8 personnes utilisent Linux. En ce qui concerne les navigateurs, Firefox est le plus populaire. Il est cité par 52% des répondants. Chrome vient ensuite avec 23% des répondants qui l'utilisent.


```{table} Les systèmes d'exploitation utilisés
:name: os_systeme

| q29_os_sys   |   nb |     freq |
|:-------------|-----:|---------:|
| Apple        |   53 | 44,5     |
| ChromeOS     |    4 |  3,4     |
| Linux        |    8 |  6,7     |
| Windows      |   54 | 45,4     |
| Total        |  119 | 100,0    |

```

Enfin, 68 répondants (soit 57%) disent avoir recourt à l'intelligence artificielle pour leurs recherches, dont 65% emploient des versions gratuites. La même proportion cite Chat GPT. Il est l'outil le plus utilisé. L'analyse de la question "pourquoi les utilisez-vous ?" à l'aide de l'algorithme de Topic modeling "BERTopic" {cite:p}`grootendorstBERTopicNeuralTopic2022` fait ressortir six motifs avancés par les personnes enquêtées pour expliquer leur utilisation de l'IA. Celui qui revient le plus souvent est le "gain de temps", que ce soit pour rédiger des e-mails, effectuer des tâches administratives ou rechercher de l'information :

> "Pour gagner du temps sur des procédures routinières ou pour formaliser des courriels ou autres plus rapidement"

> "Pour gagner du temps dans la recherche informations"

Les deux autres principaux motifs sont l'aide à la traduction et le traitement des données à travers la retranscription d'entretiens avec Whisper ou l'édition de code informatique.

> "relecture de l'anglais principalement"

> M'aider à écrire du code de traitement de donnée principalement

> analyse et visualisation des données

> analyse d'entretiens, recherche dans de multiples entretiens

L'IA est enfin utilisée pour la relecture et la correction de texte, dans un contexte pédagogique, que ce soit pour préparer des cours ou contrôle son utilisation par les étudiants, ou pour comprendre ce que ces outils font.

> Relectures, améliorations du manuscrit

> Enseignement: Structuration de cours, élaboration de syllabus, idées d'activités pédagogiques

> J'essaye de comprendre comment ils fonctionnent


Bien sûr, ces motifs peuvent apparaître au sein d'une même réponse. Par exemple, un des répondants a écrit : "NoScribe en local pour des **transcriptions d'entretiens** + deepl pour des **traductions courtes et ponctuelles**". 


```{table} Les 5 premiers outils IA les plus cités. Les pourcentages sont établis sur les 65 personnes ayant recourt à l'IA
:name: top_5_ia

| Outils IA          |   nb |   total |     freq |
|:-------------------|-----:|--------:|---------:|
| chatgpt            |   42 |      65 | 64,6     |
| claude             |   19 |      65 | 29,2     |
| deepl              |    7 |      65 | 10,8     |
| gemini             |    7 |      65 | 10,8     |
| mistral            |    6 |      65 |  9,2     |

```
```{figure} ./viz/raison_ia_bertopic.png
:name: q37_raison_ia

6 raisons d'utiliser l'IA
```

In [11]:
#On compte le nombre de choix par personne

nb_choix = df_exp.groupby(["q45_clé"]).agg(nb=("q26_env_travail", "size")).reset_index()

np.mean(nb_choix.nb)
nb_choix.groupby(["nb"]).agg(freq=("q45_clé", "size")).reset_index()

,nb,freq
0,1,51
1,2,46
2,3,22


In [12]:
### Traitement des questions ordonnées : on calcule la fréquence par rang pour chaque option, puis le rang moyen
list_rank= []

for cle in df_exp.q45_clé.unique():
    dtmp = df_exp.loc[df_exp.q45_clé==cle]
    for n, r in enumerate(dtmp.q26_env_travail):
        dict_rank={"q45_clé":cle,
                   "q26_env_travail":r,
                   "rang":n+1}
        list_rank.append(dict_rank)
list_rank
    
df_rank = pd.DataFrame.from_dict(list_rank)  

df_rank

,q45_clé,q26_env_travail,rang
0,PS8W-TJ9P,A mon domicile,1
1,GNEG-5QXQ,A mon domicile,1
2,FCCB-GMYA,A mon domicile,1
3,FCCB-GMYA,Dans mon bureau sur site,2
4,CV3G-95CT,Dans mon bureau sur site,1
...,...,...,...
204,HRZG-6JGE,A mon domicile,2
205,32TW-JLY5,A mon domicile,1
206,32TW-JLY5,Dans mon bureau sur site,2
207,6GVE-N5NJ,A mon domicile,1


In [13]:
d_class = df_rank.groupby(["rang","q26_env_travail"]).agg(nb=("q45_clé","size")).reset_index()
d_class["freq"] = round(d_class.nb/119*100, 1)
d_class_tab = pd.pivot(d_class, index= "rang", columns="q26_env_travail", values="freq").reset_index()
print(d_class_tab[["rang","A mon domicile",  "Dans mon bureau sur site",  "Autre, précisez"]].to_markdown(index=False))
#df_rank.groupby(["q26_env_travail","rang"]).agg(nb=("q45_clé","size"))

|   rang |   A mon domicile |   Dans mon bureau sur site |   Autre, précisez |
|-------:|-----------------:|---------------------------:|------------------:|
|      1 |             71.4 |                       24.4 |               4.2 |
|      2 |             16.8 |                       30.3 |              10.1 |
|      3 |              2.5 |                        4.2 |              11.8 |


In [18]:
df0.loc[df0.q26_1_autre_rec=="Autres sites (CNRS, partenaires)", "q26_1_autre_rec"] = "Autres sites (Campus Condorcet, CNRS, partenaires)"
df_exp, gb_data = split_multiple_choices(df0.loc[~df0.q26_1_autre_rec.isna()], "q26_1_autre_rec", "q45_clé", sep = '|')
gb_data

,q26_1_autre_rec,nb,total,freq,total_freq
0,Bibliothèque,7,20,35.0,100.0
1,"Autres sites (Campus Condorcet, CNRS, partenai...",6,20,30.0,100.0
2,Où je peux,5,20,25.0,100.0
3,Déplacement,1,20,5.0,100.0
4,Laboratoire,1,20,5.0,100.0
5,Métro,1,20,5.0,100.0


In [100]:
df0.q29_os_sys
print(grouped_question(df0, "q34_ia_for_research", index = "q45_clé").drop(columns=["total","total_freq"]).to_markdown(index=False))

| q34_ia_for_research   |   nb |    freq |
|:----------------------|-----:|--------:|
| Non                   |   51 | 42.8571 |
| Oui                   |   68 | 57.1429 |


In [94]:
df_exp, gb_data = split_multiple_choices(df0.loc[~df0.q30_webbrowser.isna()], "q30_webbrowser", "q45_clé", sep = '|')
gb_data

,q30_webbrowser,nb,total,freq,total_freq
0,Firefox,62,119,52.100840,100.0
1,Chrome,28,119,23.529412,100.0
2,Safari,17,119,14.285714,100.0
3,"Autre, précisez",6,119,5.042017,100.0
4,Chromium,3,119,2.521008,100.0
5,Tor browser,2,119,1.680672,100.0
6,Edge,1,119,0.840336,100.0


In [63]:
df0.q35_ia_tools.unique()
df_exp, gb_data = split_multiple_choices(df0[["q45_clé","q35_ia_tools_rec"]].loc[~df0.q35_ia_tools_rec.isna()], "q35_ia_tools_rec", "q45_clé", sep = '|')
print(gb_data.drop(columns=["total_freq"]).head(5).to_markdown(index=False))

| q35_ia_tools_rec   |   nb |   total |     freq |
|:-------------------|-----:|--------:|---------:|
| chatgpt            |   42 |      65 | 64.6154  |
| claude             |   19 |      65 | 29.2308  |
| deepl              |    7 |      65 | 10.7692  |
| gemini             |    7 |      65 | 10.7692  |
| mistral            |    6 |      65 |  9.23077 |


In [16]:
gb_data =grouped_question(df0, "q36_gratuit_ou_payant", index = "q45_clé")
gb_data["total"]=68
gb_data["freq"]= gb_data.nb/gb_data.total*100
gb_data

,q36_gratuit_ou_payant,nb,total,freq,total_freq
0,Gratuite,44,68,64.705882,100.0
1,Payante,24,68,35.294118,100.0


In [19]:
df0.q37_raison_util_ia.nunique()

67

In [18]:
for n, x in enumerate(df0.q37_raison_util_ia):
    print(df0.q45_clé.iloc[n], x)

PS8W-TJ9P nan
GNEG-5QXQ nan
FCCB-GMYA Pédagogie, Recherche (biblio)
CV3G-95CT relecture de l'anglais principalement
M5L8-UM9F aide ponctuelle formulation de traduction, aide ponctuelle réponses à des questions méthodologiques complexes
HT8V-PSKZ nan
AVQG-73T4 nan
R9R6-P7CN nan
9K9T-RVNL Principalement pour alléger ma charge de travail : rechercher des informations méthodologique ou statistiques, relire ma production en anglais, améliorer des formulations, rédiger des mails de réponse, rédiger des post professionnels pour les réseaux sociaux, etc.
CN6B-UEHQ Il s'agit de mon cœur de recherche. 
639H-EH8R nan
5G5M-BM3L pour rester au courant des progrès et fonctionnalités
9JNL-L3RA nan
BKF8-SNEZ Automatisation de tâches réccurentes, créations de site web pour des supports pédagogiques personnalisés en ligne.
YTMC-B4DP transciption d'entretien
QLPJ-DVVQ nan
RXPG-46ZW nan
Q5AZ-855W analyse d'entretiens, recherche dans de multiples entretiens
QV7Q-KLNA nan
ZGPH-2FPH ça aide
TZYG-EKYM pratiqu